[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/03_attention_variants/03.2_mqa_gqa/lab.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/03_attention_variants/03.2_mqa_gqa/lab.ipynb)

# 3.2 Lab: MQA and GQA — The Attention Head Sharing Spectrum


This lab explores the MHA → GQA → MQA spectrum through **analytical calculations only**.
We derive KV cache memory formulas, compare head-sharing strategies, and show why GQA-8 won.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -- Model parameters (Llama-3.1 8B scale) --
d_model = 4096        # hidden dimension
n_heads = 32          # query heads
d_head = d_model // n_heads  # 128 per head
n_layers = 32         # transformer layers
dtype_bytes = 2       # FP16 = 2 bytes per element
seq_len = 4096        # context length for calculations

## 1. KV Cache Memory Formula

For each layer, we store **K** and **V** projections for every token:

In [ ]:
# -- KV heads for each attention strategy --
strategies = {
    "MHA (32 KV heads)": 32,   # every Q head has its own KV
    "GQA-8 (8 KV heads)": 8,   # 4 Q heads share 1 KV (Llama-3, Mistral-7B)
    "GQA-4 (4 KV heads)": 4,   # 8 Q heads share 1 KV
    "GQA-2 (2 KV heads)": 2,   # 16 Q heads share 1 KV
    "MQA (1 KV head)": 1,      # all 32 Q heads share 1 KV
}

def kv_cache_bytes(n_kv_heads, seq=4096):
    """Total KV cache in bytes for the full model."""
    # 2 for K+V, per layer, per token
    per_token_per_layer = 2 * n_kv_heads * d_head * dtype_bytes
    return n_layers * seq * per_token_per_layer

# -- Print comparison table --
print(f"{"Strategy":<22} {"KV heads":>8} {"Per token (all layers)":>22} {"Full cache (4K ctx)":>20}")
print("-" * 76)
for name, kv_h in strategies.items():
    per_tok_all_layers = n_layers * 2 * kv_h * d_head * dtype_bytes
    total = kv_cache_bytes(kv_h)
    print(f"{name:<22} {kv_h:>8} {per_tok_all_layers/1024:>19.1f} KB {total/1024**2:>17.1f} MB")

In [ ]:
# -- Visualize KV cache reduction --
names = list(strategies.keys())
kv_heads_list = list(strategies.values())
cache_mb = [kv_cache_bytes(h) / 1024**2 for h in kv_heads_list]

fig_2, ax_2 = plt.subplots(figsize=(9, 5))
colors = ['#ef4444', '#f59e0b', '#22c55e', '#06b6d4', '#8b5cf6']
bars_kv = ax_2.bar(names, cache_mb, color=colors, edgecolor='black', linewidth=0.8)

# Annotate each bar
for bar, mb in zip(bars_kv, cache_mb):
    ax_2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{mb:.0f} MB', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax_2.set_ylabel('KV Cache Size (MB)', fontsize=12)
ax_2.set_title('KV Cache Memory at 4096 Tokens (32-layer, d=4096, FP16)', fontsize=13)
ax_2.set_ylim(0, max(cache_mb) * 1.15)
ax_2.spines['top'].set_visible(False)
ax_2.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()
# MHA uses 32x more KV memory than MQA. GQA-8 gives 4x reduction.

## 2. Head Sharing Visually

```
MHA:   Q₁→KV₁  Q₂→KV₂  Q₃→KV₃  ... Q₃₂→KV₃₂   (32 KV heads)
GQA-8: Q₁Q₂Q₃Q₄→KV₁  Q₅Q₆Q₇Q₈→KV₂  ...        (8 KV heads, 4:1 ratio)
MQA:   Q₁Q₂Q₃...Q₃₂→KV₁                          (1 KV head, 32:1 ratio)
```

**Key insight**: Query heads are cheap (not cached). KV heads are expensive
(cached for every token). Sharing KV heads across multiple Q heads cuts cache
without reducing the model's ability to ask diverse questions.

In [ ]:
# -- KV cache scaling with sequence length --
seq_lengths = np.array([512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072])

fig_3, ax_3 = plt.subplots(figsize=(9, 5))
for name, kv_h in [("MHA (32)", 32), ("GQA-8", 8), ("GQA-4", 4), ("MQA (1)", 1)]:
    cache_gb = np.array([kv_cache_bytes(kv_h, s) for s in seq_lengths]) / 1024**3
    ax_3.plot(seq_lengths / 1024, cache_gb, 'o-', label=name, linewidth=2, markersize=5)

ax_3.set_xlabel('Sequence Length (K tokens)', fontsize=12)
ax_3.set_ylabel('KV Cache (GB)', fontsize=12)
ax_3.set_title('KV Cache Growth with Context Length', fontsize=13)
ax_3.legend(fontsize=11)
ax_3.set_xscale('log', base=2)
ax_3.set_xticks([0.5, 1, 2, 4, 8, 16, 32, 64, 128])
ax_3.set_xticklabels(['0.5K', '1K', '2K', '4K', '8K', '16K', '32K', '64K', '128K'])
ax_3.grid(True, alpha=0.3)
ax_3.spines['top'].set_visible(False)
ax_3.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()
# At 128K tokens, MHA needs ~16 GB for KV alone. GQA-8 needs ~4 GB.

## 3. The MQA Tradeoff: Why Nobody Uses It Anymore

MQA collapses all KV heads into **one**. Maximum memory savings, but:

| Metric | MHA | GQA-8 | MQA |
|--------|-----|-------|-----|
| KV cache size | 1x | 0.25x | 0.03x |
| Perplexity degradation | baseline | ~0.1-0.3% | ~1-2% |
| Training stability | stable | stable | harder to converge |
| Adoption (2024+) | legacy | **dominant** | abandoned |

**Why MQA failed:**
1. **Quality gap is real** — 1-2% perplexity loss compounds across long generations
2. **Diminishing returns** — going from 8→1 KV heads saves 8x, but GQA-8 already fits in GPU memory
3. **Training instability** — single KV head becomes a gradient bottleneck
4. **GQA is free** — uptraining MHA→GQA-8 takes only 5% of original compute (Llama-2 paper)

In [ ]:
# -- Batch size capacity: concurrent users on A10G (24 GB) --
gpu_vram_gb = 24.0       # A10G
model_weights_gb = 14.0  # ~7B params in FP16
available_for_kv_gb = gpu_vram_gb - model_weights_gb  # 10 GB for KV

kv_options = [("MHA (32)", 32), ("GQA-8", 8), ("GQA-4", 4), ("MQA (1)", 1)]
max_seqs = []
for name, kv_h in kv_options:
    cache_per_seq_gb = kv_cache_bytes(kv_h, seq_len=4096) / 1024**3
    max_seqs.append(int(available_for_kv_gb / cache_per_seq_gb))

fig_4, ax_4 = plt.subplots(figsize=(9, 5))
colors_batch = ['#ef4444', '#22c55e', '#06b6d4', '#8b5cf6']
bars_users = ax_4.bar([n for n, _ in kv_options], max_seqs, color=colors_batch, edgecolor='black')
for bar, val in zip(bars_users, max_seqs):
    ax_4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(val), ha='center', fontsize=13, fontweight='bold')

ax_4.set_ylabel('Max Concurrent Sequences', fontsize=12)
ax_4.set_title('Batch Capacity on A10G (24 GB) \u2014 7B Model, 4K Context', fontsize=13)
ax_4.spines['top'].set_visible(False)
ax_4.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()
# GQA-8 serves ~4x more concurrent users than MHA on the same hardware.

## 4. Why GQA-8 Won

The sweet spot is clear:

1. **4x KV cache reduction** — enough to quadruple batch size on any GPU
2. **Negligible quality loss** — <0.3% perplexity, undetectable in human evals
3. **Easy adoption** — existing MHA checkpoints uptrain to GQA in 5% compute
4. **Universal** — Llama-3, Mistral-7B, Gemma, Qwen-2, DeepSeek all use GQA

The 4:1 ratio (32Q / 8KV) balances:
- Enough KV diversity for different positional relationships
- Enough sharing to dramatically cut memory bandwidth during decode

In [ ]:
# -- Cost-per-token comparison (derived from batch capacity) --
gpu_cost_per_hr = 0.50  # $/hr for A10G
tokens_per_sec_per_seq = 50  # decode speed per sequence

fig_5, ax_5 = plt.subplots(figsize=(9, 5))
cost_per_m = []
for (name, kv_h), batch in zip(kv_options, max_seqs):
    throughput = max(batch, 1) * tokens_per_sec_per_seq  # tokens/sec total
    cpm = (gpu_cost_per_hr / 3600 / throughput) * 1e6    # $/M tokens
    cost_per_m.append(cpm)

bars_cost = ax_5.bar([n for n, _ in kv_options], cost_per_m, color=colors_batch, edgecolor='black')
for bar, val in zip(bars_cost, cost_per_m):
    ax_5.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.05,
            f'${val:.3f}', ha='center', fontsize=11, fontweight='bold')

ax_5.set_ylabel('Cost per Million Tokens ($)', fontsize=12)
ax_5.set_title('Serving Cost: KV Head Sharing Directly Reduces $/Token', fontsize=13)
ax_5.spines['top'].set_visible(False)
ax_5.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## Key Takeaways

| | MHA | GQA-8 | MQA |
|---|---|---|---|
| KV heads | 32 | 8 | 1 |
| Cache per token (all layers) | 524 KB | 131 KB | 16 KB |
| Cache at 4K context | 2048 MB | 512 MB | 64 MB |
| Batch capacity (A10G, 7B) | 4 | 19 | 156 |
| Quality loss | 0% | <0.3% | 1-2% |
| Industry status (2024+) | legacy | **standard** | deprecated |

**The lesson**: Reducing from 32→8 KV heads captures most of the memory savings
while preserving nearly all model quality. This is why every major LLM since 2023 uses GQA.